[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/gauravs19/iiot-predictive-maintenance/blob/main/notebooks/01_eda_and_features.ipynb)

# 01 · Exploratory Data Analysis & Feature Engineering

**Goal:** understand the sensor signals and turn raw readings into features the
models can learn from. Good features matter more than fancy models — this notebook
is where most of the predictive power actually comes from.

We cover:
1. Which C-MAPSS sensors carry a usable degradation signal (and which are dead).
2. Building the **RUL label** for supervised training.
3. **Rolling-window features** that capture *trends*, not just instantaneous values.
4. Reshaping the time-series into **sequences** for the LSTM.

## Step 1 · Bootstrap + imports

In [ ]:
# --- Environment bootstrap (works locally AND on Google Colab) ---------------
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # On Colab there is no repo yet, so clone it and install dependencies.
    !git clone -q https://github.com/gauravs19/iiot-predictive-maintenance.git
    %cd iiot-predictive-maintenance
    !pip install -q -r requirements.txt

# Make the repo root importable so `from src import ...` works from notebooks/.
def _find_repo_root(start="."):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, "src")):
            return p
        p = os.path.dirname(p)
    raise RuntimeError("repo root (folder containing src/) not found")

REPO_ROOT = _find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)
print("running on Colab" if IN_COLAB else "running locally")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import data, features
print("imports OK")

## Step 2 · Load the data again

Because notebooks run independently, we reload C-MAPSS here. The download is cached
on disk from notebook `00`, so this is instant the second time.

In [ ]:
cmapss = data.load_cmapss("FD001")
train = cmapss["train"].copy()
print("train shape:", train.shape)

## Step 3 · Which sensors actually carry information?

C-MAPSS has 21 sensors, but in the FD001 operating regime several are **flat lines**
— they never change, so they can't help predict anything. Feeding constant columns
to a model just adds noise and slows training.

Below we compute the standard deviation of each sensor. Sensors with (near-)zero
variance are "dead". Our `features.feature_columns()` helper drops a known list of
these so every model uses the same clean inputs.

In [ ]:
sensor_cols = [c for c in train.columns if c.startswith("sensor_")]
std = train[sensor_cols].std().sort_values()
print("Lowest-variance (likely dead) sensors:")
print(std.head(8).round(4))
print("\nActive feature columns used downstream:")
print(features.feature_columns(train))

## Step 4 · Visualise sensor degradation over an engine's life

The core intuition behind predictive maintenance: as a machine wears out, its
sensor readings **drift**. Below we plot a few informative sensors for a single
engine against its cycle number. You should see clear upward/downward trends as the
engine approaches failure — that drift is the signal our models exploit.

In [ ]:
unit1 = train[train["unit"] == 1]
show = ["sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_11", "sensor_15"]
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, s in zip(axes.ravel(), show):
    ax.plot(unit1["cycle"], unit1[s])
    ax.set_title(s); ax.set_xlabel("cycle")
fig.suptitle("Engine #1 — sensor drift toward failure", y=1.02)
fig.tight_layout(); plt.show()

## Step 5 · Build the RUL (Remaining Useful Life) label

For supervised training we need a target. For each row, **RUL = (engine's last
cycle) − (current cycle)** — i.e. how many cycles remain before failure.

**One subtlety (and why we clip):** early in an engine's life there's no visible
degradation, so a literally-correct RUL of, say, 300 is unlearnable from sensors
that look perfectly healthy. The standard C-MAPSS convention is a **piecewise-linear
RUL capped at ~125**: we treat anything healthier than 125 cycles-to-go as "125+".
This matches the physics (degradation only becomes observable near end-of-life) and
trains far better. `features.add_rul()` does exactly this.

In [ ]:
train = features.add_rul(train, clip=125)
print("RUL range after clipping:", train["rul"].min(), "to", train["rul"].max())

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(train[train.unit == 1]["cycle"], train[train.unit == 1]["rul"])
ax.set_title("Clipped RUL target for engine #1 (flat at 125, then linear to 0)")
ax.set_xlabel("cycle"); ax.set_ylabel("RUL"); plt.show()

## Step 6 · Rolling-window features (capturing *trend*)

A single sensor reading is a snapshot; degradation is about **change over time**. We
add, per engine, a **rolling mean** (smooths noise, shows the trend) and **rolling
standard deviation** (rising variance often precedes failure) over a short window.

`features.add_rolling_features()` computes these *per unit* so one engine's history
never leaks into another's.

In [ ]:
cols = features.feature_columns(train)
train_fe = features.add_rolling_features(train, cols, window=5)
new_cols = [c for c in train_fe.columns if c.endswith(("_rmean", "_rstd"))]
print(f"Added {len(new_cols)} rolling features. Example new columns:")
print(new_cols[:6])
train_fe.filter(regex="sensor_2(_rmean|_rstd)?$").head()

## Step 7 · Reshape into sequences for the LSTM

Tree models (notebook `02`, AI4I) take a flat row of features. But an **LSTM** learns
from *ordered sequences*, so we slide a fixed-length window (here 30 cycles) over each
engine's history. Every window becomes one training example shaped
`(timesteps=30, features)`, labelled with the RUL at the **end** of the window.

`features.make_sequences()` returns a 3-D array `(n_windows, 30, n_features)` — the
exact shape PyTorch's LSTM expects.

In [ ]:
X_seq, y_seq = features.make_sequences(train_fe, cols, seq_len=30, label="rul")
print("Sequence tensor X:", X_seq.shape, "  (windows, timesteps, features)")
print("Label vector   y:", y_seq.shape)
print("Example label (RUL at end of first window):", y_seq[0])

## Step 8 · Recap

We now have:
- A **clean feature set** (dead sensors removed).
- A **clipped RUL label** suitable for regression.
- **Rolling features** capturing degradation trend.
- **Sequence tensors** ready for the LSTM.

**Next:** notebook `02` trains the predictive-maintenance models — a classifier on
AI4I and an LSTM RUL regressor on C-MAPSS — and explains each metric.